# Fine-tune Qwen2.5-3B for Legal Case Analysis

This notebook fine-tunes Qwen2.5-3B-Instruct on Sri Lankan legal case data using Unsloth for efficient training.

## Dataset
- **Training examples**: 314 real legal cases
- **Format**: Instruction-tuning with OCR text input and structured JSON output
- **Source**: Supreme Court and Court of Appeal cases

## Step 1: Install Dependencies

In [ ]:
%%capture
# Install Unsloth for efficient fine-tuning
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install compatible versions
!pip install "peft>=0.13.0" --upgrade
!pip install --no-deps trl accelerate bitsandbytes
!pip install datasets

print("✅ Installation complete!")

## Step 2: Check GPU and Setup

In [ ]:
import torch
import json
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected - training will be slow")

## Step 3: Load Base Model

In [ ]:
from unsloth import FastLanguageModel

# Configuration
max_seq_length = 4096  # Adjust based on document lengths
dtype = None  # Auto-detect
load_in_4bit = True  # Memory efficiency

# Load model
print("🔄 Loading Qwen2.5-3B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Model loaded successfully!")
print(f"Model parameters: {model.get_memory_footprint() / 1e9:.1f} GB")

## Step 4: Configure LoRA (Low-Rank Adaptation)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # Rank - higher = more parameters but better quality
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.1,  # Dropout for regularization
    bias="none",  # No bias adaptation
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=42,
)

print("✅ LoRA adapters configured!")
print(f"Trainable parameters: {model.get_nb_trainable_parameters()}")

## Step 5: Load and Prepare Training Data

In [ ]:
# Load the dataset
DATASET_PATH = "/kaggle/input/civil-cases-training-data/civil_cases.jsonl"

print(f"📁 Loading dataset from: {DATASET_PATH}")

# Read JSONL file
data = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"✅ Loaded {len(data)} training examples")

# Check first example
print("\n📄 Sample training example:")
sample = data[0]
print(f"Instruction: {sample['instruction'][:100]}...")
print(f"Input length: {len(sample['input'])} characters")
print(f"Output length: {len(sample['output'])} characters")

## Step 6: Format Data for Training

In [ ]:
# Chat template for Qwen
chat_template = """<|im_start|>system
You are an expert legal AI assistant specializing in Sri Lankan law.<|im_end|>
<|im_start|>user
{instruction}

{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

def format_example(example):
    """Format a single example for training."""
    return chat_template.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"]
    )

# Apply formatting
formatted_data = []
for example in data:
    formatted_text = format_example(example)
    formatted_data.append({"text": formatted_text})

# Create dataset
dataset = Dataset.from_list(formatted_data)

print(f"✅ Formatted {len(dataset)} examples for training")
print(f"Average text length: {sum(len(x['text']) for x in formatted_data) / len(formatted_data):.0f} characters")

## Step 7: Training Configuration

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,  # Number of training epochs
    per_device_train_batch_size=1,  # Batch size per GPU
    gradient_accumulation_steps=4,  # Effective batch size = 1 * 4 = 4
    optim="adamw_8bit",  # 8-bit optimizer for memory efficiency
    learning_rate=2e-4,  # Learning rate
    weight_decay=0.01,  # Regularization
    fp16=True,  # Mixed precision training
    bf16=False,
    max_grad_norm=1.0,  # Gradient clipping
    max_steps=-1,
    warmup_ratio=0.1,  # Learning rate warmup
    group_by_length=True,  # Group similar length samples
    lr_scheduler_type="cosine",  # Learning rate schedule
    logging_steps=10,  # Log every N steps
    save_steps=100,  # Save checkpoint every N steps
    save_total_limit=2,  # Keep only last 2 checkpoints
    evaluation_strategy="no",  # No evaluation during training
    seed=42,
)

print("✅ Training configuration ready")

## Step 8: Initialize Trainer

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=False,  # Don't pack multiple samples
    args=training_args,
)

print("✅ Trainer initialized")
print(f"📊 Training dataset size: {len(dataset)}")
print(f"🔄 Total training steps: {trainer.state.max_steps if trainer.state.max_steps > 0 else len(dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

## Step 9: Start Training! 🚀

In [ ]:
import time
start_time = time.time()

print("🚀 Starting training...")
print("📊 Progress will be shown below:")
print("" + "="*50)

# Start training
trainer.train()

# Training completed
end_time = time.time()
training_time = (end_time - start_time) / 60  # Convert to minutes

print("" + "="*50)
print(f"✅ Training completed!")
print(f"⏱️ Training time: {training_time:.1f} minutes")
print(f"📈 Final loss: {trainer.state.log_history[-1].get('train_loss', 'N/A')}")

## Step 10: Save the Fine-tuned Model

In [ ]:
# Save LoRA adapters
print("💾 Saving LoRA adapters...")
model.save_pretrained("civil_legal_model_lora")
tokenizer.save_pretrained("civil_legal_model_lora")

print("✅ LoRA adapters saved to 'civil_legal_model_lora/'")
print("📁 Files created:")
import os
for file in os.listdir("civil_legal_model_lora"):
    print(f"   - {file}")

## Step 11: Test the Fine-tuned Model

In [ ]:
# Test with a sample legal document
test_instruction = "Extract structured legal data from the OCR text of a Sri Lankan Supreme Court judgment. Return ONLY a single valid JSON object matching the required schema. Use null for missing fields and do not invent facts."

test_input = """IN THE SUPREME COURT OF THE DEMOCRATIC SOCIALIST REPUBLIC OF SRI LANKA

SC Appeal No. 123/2022

Between:
John Doe - Appellant
vs.
Attorney General - Respondent

Date: 2023-06-15

This is a test case for demonstration purposes."""

# Format for inference
test_prompt = f"""<|im_start|>system
You are an expert legal AI assistant specializing in Sri Lankan law.<|im_end|>
<|im_start|>user
{test_instruction}

{test_input}<|im_end|>
<|im_start|>assistant"""

# Tokenize and generate
FastLanguageModel.for_inference(model)  # Enable inference mode
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

print("🧪 Testing fine-tuned model...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
assistant_response = response.split("<|im_start|>assistant")[-1].strip()

print("🎯 Model Response:")
print("" + "-"*50)
print(assistant_response)
print("" + "-"*50)

## Step 12: Training Summary

In [ ]:
print("🎉 TRAINING COMPLETE!")
print("=" * 50)
print(f"📊 Dataset: {len(data)} legal cases")
print(f"🤖 Base model: Qwen2.5-3B-Instruct")
print(f"🔧 Method: LoRA fine-tuning")
print(f"📈 Epochs: {training_args.num_train_epochs}")
print(f"⏱️ Training time: {training_time:.1f} minutes")
print(f"💾 Model saved: civil_legal_model_lora/")
print("\n✅ Your specialized legal AI model is ready!")
print("📋 Next steps:")
print("   1. Download the model files")
print("   2. Integrate with your application")
print("   3. Test with real legal documents")